In [1]:
from pyspark.sql import functions as F

# Read cleaned OpenAQ data from Silver Lakehouse
air_df = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Silver_LH.dbo.openaq_pm25_daily_clean"
)

print("Silver records:", air_df.count())

# Build daily air quality fact table
gold_air_df = (
    air_df
    .groupBy(
        "date",
        "station",
        "location_id",
        "sensor_id",
        "parameter",
        "unit"
    )
    .agg(
        F.avg("value").alias("avg_value"),
        F.min("value").alias("min_value"),
        F.max("value").alias("max_value"),
        F.avg("coverage_percent").alias("avg_coverage_percent"),
        F.count("*").alias("record_count")
    )
    .withColumn("date", F.to_date("date"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
)

# Save to Gold Lakehouse
(
    gold_air_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("FactAirQualityDaily")
)

print("Gold Air Quality table saved successfully!")
print("Gold records:", gold_air_df.count())

display(gold_air_df.orderBy("date"))

StatementMeta(, 440eb133-0f3f-48f2-8c7e-617172339db7, 3, Finished, Available, Finished, False)

Silver records: 31
Gold Air Quality table saved successfully!
Gold records: 31


SynapseWidget(Synapse.DataFrame, 5b3cede4-96e8-4938-8106-d9563aeaddfe)